# Jour 2 · Chapitre 2 — Scénario : le stress test du desk Risque
## Formation Beobank · SAS → Python · Cahier participant

**L'histoire.** La BNB (Banque Nationale de Belgique) demande à Beobank un mini stress test sur
les soldes de comptes. Élodie, responsable du desk Risque, vous explique :
*"J'ai besoin de chiffres rapides sur les soldes : moyenne, dispersion, découverts, agios, et
l'impact de trois scénarios de taux. Pas de boucle `for` : tout doit être vectorisé avec NumPy,
on refera le calcul sur des millions de comptes."*

**Comment travailler.** Remplacez chaque `______`. Les commentaires indiquent la notion NumPy à
utiliser ; retrouvez la fonction exacte dans le cours.

📎 **Matière à relire :** `Cours_NumPy.ipynb` (même dossier) — création de tableaux, types,
copy vs view, shape/reshape, opérations vectorisées, filtres booléens, statistiques, tri.

## Préparation (déjà écrite, exécutez-la)

In [ ]:
# ── Extraction des soldes en tableau NumPy ────────────────────────────────
import numpy as np
import pandas as pd

ctr = pd.read_csv("../../data/CTR.csv", sep=";", na_values=".", encoding="utf-8")
soldes_brut = ctr["SLD_CTR"].to_numpy()       # tableau NumPy de 200 soldes, avec des NaN
print(type(soldes_brut), soldes_brut.shape)

---
## Mission 1 — Nettoyer le tableau

*"Les soldes inconnus faussent tout. Retire-les avant de calculer."*

In [ ]:
# ── Filtrer les NaN ───────────────────────────────────────────────────────
# Quelle fonction NumPy renvoie True là où la valeur est NaN ? (~ inverse le masque)
soldes = soldes_brut[~np.______(soldes_brut)]
# Quel attribut donne le nombre total d'éléments d'un tableau ?
print("Soldes connus :", soldes.______)          # → 63
# Quel attribut indique le type des éléments stockés ?
print("Type :", soldes.______)                  # → float64

---
## Mission 2 — Les statistiques de base

*"Moyenne, médiane, écart-type, et le 5e percentile — c'est ce que la BNB regarde."*

In [ ]:
# ── Statistiques NumPy ────────────────────────────────────────────────────
print("Moyenne      :", np.______(soldes).round(2))
# Quelle statistique coupe la distribution en deux moitiés égales ?
print("Médiane      :", np.______(soldes))
# Quelle méthode d'un tableau mesure la dispersion autour de la moyenne ?
print("Écart-type   :", soldes.______().round(2))
# Quelle fonction donne la valeur sous laquelle se trouvent 5 % des soldes ?
print("Percentile 5 :", np.______(soldes, 5).round(2))
# Quelles fonctions donnent le solde le plus bas et le plus haut ?
print("Min / Max    :", np.______(soldes), "/", np.______(soldes))

✍️ **À vous :** la moyenne est très négative alors que la médiane vaut 0. Qu'est-ce que ça révèle
sur la distribution des soldes ?

---
## Mission 3 — Les comptes à découvert

*"Combien de comptes sont dans le rouge, et pour quel montant total ?"*

In [ ]:
# ── Masques booléens ──────────────────────────────────────────────────────
# Quelle comparaison crée un masque True pour les soldes négatifs ?
decouvert = soldes ______ 0
# Additionner un masque True/False compte les True : quelle méthode ?
print("Comptes à découvert :", decouvert.______())                    # → 13
# On indexe le tableau avec le masque lui-même pour ne garder que les découverts
print("Total des découverts :", soldes[______].sum().round(2))

---
## Mission 4 — Calculer les agios du mois

*"Taux débiteur : 9,5 % par an. Calcule les agios mensuels de chaque compte, 0 si créditeur."*

In [ ]:
# ── np.where vectorisé ────────────────────────────────────────────────────
TAUX_DEBITEUR = 0.095
# Quelle fonction NumPy choisit entre deux valeurs selon une condition, élément par élément ?
# Pour un compte créditeur, les agios valent combien ?
agios = np.______(soldes < 0, -soldes * TAUX_DEBITEUR / 12, ______)
# Arrondir au centime : combien de décimales ?
agios = agios.round(______)
print("Agios du mois :", agios.sum().round(2), "EUR")

---
## Mission 5 — Trois scénarios de taux

*"Simule les intérêts créditeurs mensuels à 1 %, 2 % et 3,5 % — en une seule opération."*

Idée : transformer le vecteur des soldes en **colonne** (63 × 1), puis le multiplier par le
vecteur des taux (3) → tableau 63 × 3 grâce au *broadcasting*.

In [ ]:
# ── Broadcasting ──────────────────────────────────────────────────────────
crediteurs = soldes[soldes > 0]
taux = np.array([0.01, 0.02, 0.035])
# Quel objet NumPy ajoute un axe pour transformer un vecteur en colonne ?
interets = crediteurs[:, ______] * taux / 12
# Quel attribut donne les dimensions du résultat ?
print("Dimensions :", interets.______)
# On veut le total PAR SCÉNARIO (par colonne) : quel axe additionner ?
print("Intérêts par scénario :", interets.sum(axis=______).round(2))

---
## Mission 6 — Le piège de la vue

*"J'ai mis à zéro mes 3 premiers soldes pour tester... et le tableau d'origine a changé !"*

In [ ]:
# ── copy vs view ──────────────────────────────────────────────────────────
# Une tranche de tableau est une VUE : quelle méthode crée un tableau indépendant ?
test = soldes[:3].______()
test[:] = 0
print("Test :", test, "| Origine intacte :", soldes[:3])

---
## 🏁 Mission finale — Le top 3 des risques pour la BNB

*"Donne-moi les 3 comptes les plus débiteurs, avec leur solde et leurs agios."*

In [ ]:
# ── Tri par indices ───────────────────────────────────────────────────────
identifiants = ctr["IDT_AC"].to_numpy()[~np.isnan(soldes_brut)]   # identifiants alignés sur `soldes`
# Quelle fonction renvoie les POSITIONS qui trieraient le tableau (du plus petit au plus grand) ?
pires = np.______(soldes)[:3]
print("TOP 3 DES RISQUES")
for rang, i in enumerate(pires, start=1):
    print(f"  {rang}. compte {identifiants[i]} | solde {soldes[i]:>12,.2f} | agios {agios[i]:>8,.2f}")

✍️ **Ce que ça veut dire pour la banque :** quelle part du total des découverts représentent ces
3 comptes ? Quelle action proposeriez-vous ?